# Phase 8 - Evaluation (two axes: faithfulness + correctness, + ROUGE-L proxy + HITL)

Scores the **frozen** Phase-7 generations on **two distinct axes**, because Phase 7 showed the
generator answers from whatever it retrieves (it grounds in the wrong chunk when the gold chunk
is missing):

- **Faithfulness (vs CONTEXT)** — intrinsic hallucination. gpt-4o judge sees query+context+answer,
  *never the gold answer*. Claim-level, computed from per-claim labels. Expected ~flat across
  retrievers (the model rarely invents beyond its context).
- **Correctness (vs GOLD)** — answer accuracy. A **separate** gpt-4o pass sees query+gold-chunk+
  answer, *never the context*. This is where the retrieval→generation coupling lives: when the
  gold chunk isn't retrieved, the model is faithful-to-context but **incorrect**.
- **ROUGE-L** (proxy, gold_in_context==True only) and **HITL** (validates both axes).

**Hash gates first:** asserts the golden hash AND the Phase-7 generations hash. Abstentions are a
separate category (neither judge runs). Descriptive only (no significance tests).

In [1]:
%load_ext autoreload
%autoreload 2

import sys, json, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
print("pandas", pd.__version__)

CHUNKS_PATH  = ROOT / "data" / "processed" / "chunks_n200.parquet"
GOLDEN_PATH  = ROOT / "data" / "processed" / "golden_queries.parquet"
GEN_PATH     = ROOT / "data" / "processed" / "phase_07_generations.parquet"
SCORES_PATH  = ROOT / "data" / "processed" / "phase_08_scores.parquet"
SUMMARY_PATH = ROOT / "notes" / "phase_08_summary.json"
HITL_PATH    = ROOT / "notes" / "phase_08_hitl_sheet.csv"
FIG_DIR      = ROOT / "reports" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

GOLDEN_SHA256      = "325634f065147a49d6fba6e0fb021107d536b1aa717bcbbb4a10b68b93c0e72e"
GENERATIONS_SHA256 = "862cf6931f3b8fd88c920a7c98c53cda5412c9617f5472cafefe958fcada5ab1"
JUDGE_MODEL = "gpt-4o" 

pandas 3.0.2


## 1. Load + assert BOTH hashes (HARD GATES)

In [2]:
from src.retrieval.bm25_retriever import load_chunks
from src.retrieval.golden_dataset_builder import load_golden_set, hash_golden_set
from src.generation.generator import load_generations
from src.evaluation.evaluator import assert_generations_hash, build_gold_text_lookup

chunks_df = load_chunks(CHUNKS_PATH)
golden_df = load_golden_set(GOLDEN_PATH)
gen_df    = load_generations(GEN_PATH)

assert hash_golden_set(golden_df) == GOLDEN_SHA256, "golden hash mismatch"
print("golden hash assert: PASS")
assert_generations_hash(gen_df, GENERATIONS_SHA256)
print("generations hash assert: PASS")

gold_text_by_chunk = build_gold_text_lookup(chunks_df)
print(f"generations: {len(gen_df)} rows | gold-text lookup: {len(gold_text_by_chunk)} chunks")

golden hash assert: PASS
generations hash assert: PASS
generations: 180 rows | gold-text lookup: 21050 chunks


## 2. Wire BOTH judges (faithfulness vs context, correctness vs gold)

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
from src.evaluation.evaluator import (make_openai_judge_fn, make_openai_correctness_fn,
                                      JUDGE_SYSTEM_PROMPT, CORRECTNESS_SYSTEM_PROMPT)

load_dotenv(override=True)
client = OpenAI()
faith_fn   = make_openai_judge_fn(client, model=JUDGE_MODEL, temperature=0.0, seed=42)
correct_fn = make_openai_correctness_fn(client, model=JUDGE_MODEL, temperature=0.0, seed=42)

print("judge model:", JUDGE_MODEL)
print("\n--- FAITHFULNESS rubric (vs context) ---\n", JUDGE_SYSTEM_PROMPT)
print("\n--- CORRECTNESS rubric (vs gold) ---\n", CORRECTNESS_SYSTEM_PROMPT)

judge model: gpt-4o

--- FAITHFULNESS rubric (vs context) ---
 You are a meticulous financial-analysis evaluator. You judge whether an ANSWER is grounded in a set of numbered CONTEXT passages and whether it addresses the QUESTION. The context passages are the ONLY ground truth; do not use outside knowledge.

Procedure:
1. If the answer declines to answer or states the context is insufficient, set "is_abstention" true and return an empty "claims" list.
2. Otherwise decompose the answer into its distinct factual claims (each a single checkable assertion: a figure, date, named entity, or statement).
3. For each claim decide whether the CONTEXT supports it:
   "yes"     = directly stated or unambiguously entailed by a passage;
   "partial" = partly supported, or right topic but wrong magnitude/qualifier;
   "no"      = not supported by any passage (a hallucination w.r.t. context).
   Record the supporting passage number, or null.
4. Rate "relevance" in [0,1]: how well the answer addresses 

## 3. Smoke test - both judges on ONE answered generation

In [4]:
from src.evaluation.evaluator import score_one

probe = gen_df[~gen_df["abstained"]].iloc[0]
gold0 = gold_text_by_chunk.get(str(list(probe["relevant_chunk_ids"])[0]))
v = score_one(probe, faith_fn, correctness_fn=correct_fn, gold_text=gold0)
print("query        :", probe["query_text"][:90])
print("gold_in_context:", v["gold_in_context"])
print("FAITHFULNESS :", v["faithfulness"], "| relevance:", v["relevance"],
      "| claims:", v["n_supported"], "yes /", v["n_partial"], "partial /", v["n_unsupported"], "no")
print("CORRECTNESS  :", v["correctness"], f"({v['correctness_label']})", "|", v["correctness_note"][:80])
print("rouge_l      :", round(v["rouge_l"], 3) if v["rouge_l"] is not None else None)

query        : In the closing remarks of the call, what did the CEO thank participants for and say about 
gold_in_context: False
FAITHFULNESS : 1.0 | relevance: 1.0 | claims: 4 yes / 0 partial / 0 no
CORRECTNESS  : 0.0 (incorrect) | The answer incorrectly attributes the closing remarks to CEO Rick Smith instead 
rouge_l      : 0.155


## 4. Score all 180 on both axes (abstained rows skip both judges)

~170 answered rows × 2 passes ≈ 340 gpt-4o calls (a few dollars, a few minutes).

In [5]:
from src.evaluation.evaluator import run_evaluation

def _progress(done, total):
    if done == 1 or done % 30 == 0 or done == total:
        print(f"  {done:3d}/{total}")

t0 = time.time()
scored = run_evaluation(gen_df, faith_fn, correctness_fn=correct_fn,
                        gold_text_by_chunk=gold_text_by_chunk, progress=_progress)
print(f"\nscored {len(scored)} rows in {time.time()-t0:.1f}s")
print("faithfulness parse errors:", int(scored["judge_parse_error"].sum()),
      "| correctness parse errors:", int(scored["correctness_parse_error"].sum()), "(want 0)")

    1/180
   30/180
   60/180
   90/180
  120/180
  150/180
  180/180

scored 180 rows in 817.6s
faithfulness parse errors: 0 | correctness parse errors: 0 (want 0)


In [6]:
from src.evaluation.evaluator import save_scores
save_scores(scored, SCORES_PATH); print("saved ->", SCORES_PATH)

saved -> d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\phase_08_scores.parquet


## 5. Headline - the two-axis contrast

In [7]:
answered = scored[~scored["abstained"]]
def piv(df, col):
    return (df.groupby(["retriever", "source"])[col].mean().reset_index()
            .pivot(index="retriever", columns="source", values=col).round(3))

print("FAITHFULNESS by retriever (expect ~flat - intrinsic hallucination is low):")
print(answered.groupby("retriever")["faithfulness"].mean().round(3).to_string()); print()
print("CORRECTNESS by retriever (expect dense lowest - the retrieval coupling):")
print(answered.groupby("retriever")["correctness"].mean().round(3).to_string()); print()
print("CORRECTNESS by retriever x source:"); print(piv(answered, "correctness")); print()
print("RELEVANCE by retriever:")
print(answered.groupby("retriever")["relevance"].mean().round(3).to_string()); print()
print("ABSTENTION rate by retriever:")
print(scored.groupby("retriever")["abstained"].mean().round(3).to_string())

FAITHFULNESS by retriever (expect ~flat - intrinsic hallucination is low):
retriever
bm25      0.967
dense     0.954
hybrid    0.962

CORRECTNESS by retriever (expect dense lowest - the retrieval coupling):
retriever
bm25      0.705
dense     0.614
hybrid    0.754

CORRECTNESS by retriever x source:
source     earnings  edgar
retriever                 
bm25          0.696  0.714
dense         0.630  0.600
hybrid        0.778  0.733

RELEVANCE by retriever:
retriever
bm25      0.989
dense     0.951
hybrid    0.986

ABSTENTION rate by retriever:
retriever
bm25      0.067
dense     0.050
hybrid    0.050


In [8]:
# The sharp test of the thesis: CORRECTNESS gated by gold_in_context vs FAITHFULNESS (flat).
def by_gic(col):
    return (answered.groupby(["retriever", "gold_in_context"])[col].mean().reset_index()
            .pivot(index="retriever", columns="gold_in_context", values=col).round(3))
print("CORRECTNESS by gold_in_context  (expect False << True - answering without the gold chunk):")
print(by_gic("correctness")); print()
print("FAITHFULNESS by gold_in_context (expect ~flat - grounded either way):")
print(by_gic("faithfulness")); print()
print("ROUGE-L (proxy): overall vs gold_in_context==True subset:")
print("  overall:", answered.groupby("retriever")["rouge_l"].mean().round(3).to_dict())
print("  gold_in:", answered[answered.gold_in_context].groupby("retriever")["rouge_l"].mean().round(3).to_dict())

CORRECTNESS by gold_in_context  (expect False << True - answering without the gold chunk):
gold_in_context  False  True 
retriever                    
bm25              0.25  0.804
dense             0.25  0.782
hybrid            0.50  0.809

FAITHFULNESS by gold_in_context (expect ~flat - grounded either way):
gold_in_context  False  True 
retriever                    
bm25             0.925  0.976
dense            0.924  0.968
hybrid           0.925  0.970

ROUGE-L (proxy): overall vs gold_in_context==True subset:
  overall: {'bm25': 0.18, 'dense': 0.17, 'hybrid': 0.179}
  gold_in: {'bm25': 0.186, 'dense': 0.182, 'hybrid': 0.181}


## 6. Write `phase_08_summary.json` (real numbers)

In [9]:
from src.evaluation.evaluator import summarize_scores
summ = summarize_scores(scored)
summ["judges"] = {"model": JUDGE_MODEL, "temperature": 0.0, "seed": 42,
                  "faithfulness": "claim_level_vs_context",
                  "correctness": "separate_pass_vs_gold_chunk"}
summ["golden_set"] = {"sha256": GOLDEN_SHA256, "hash_assert": "PASS"}
summ["generations_artifact"] = {"path": str(GEN_PATH), "sha256": GENERATIONS_SHA256, "hash_assert": "PASS"}
summ["hitl"] = None
SUMMARY_PATH.write_text(json.dumps(summ, indent=2))
print(json.dumps(summ, indent=2))

{
  "n_scored": 180,
  "n_answered": 170,
  "n_abstained": 10,
  "judge_parse_errors": 0,
  "correctness_parse_errors": 0,
  "faithfulness": {
    "by_retriever": {
      "bm25": 0.9667,
      "dense": 0.9543,
      "hybrid": 0.9621
    },
    "by_retriever_source": {
      "bm25/earnings": 0.9482,
      "bm25/edgar": 0.9851,
      "dense/earnings": 0.9099,
      "dense/edgar": 0.9957,
      "hybrid/earnings": 0.954,
      "hybrid/edgar": 0.9694
    },
    "by_gold_in_context": {
      "gold_in_context_true": {
        "bm25": 0.9757,
        "dense": 0.9675,
        "hybrid": 0.97
      },
      "gold_in_context_false": {
        "bm25": 0.925,
        "dense": 0.924,
        "hybrid": 0.925
      }
    }
  },
  "correctness": {
    "by_retriever": {
      "bm25": 0.7054,
      "dense": 0.614,
      "hybrid": 0.7544
    },
    "by_retriever_source": {
      "bm25/earnings": 0.6964,
      "bm25/edgar": 0.7143,
      "dense/earnings": 0.6296,
      "dense/edgar": 0.6,
      "hybrid/earn

## 7. Figures

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np
order = ["bm25", "dense", "hybrid"]

# Fig A — the two-axis contrast (faithfulness flat vs correctness separating)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
x = np.arange(len(order)); w = 0.38
faith = [answered[answered.retriever == r]["faithfulness"].mean() for r in order]
corr  = [answered[answered.retriever == r]["correctness"].mean() for r in order]
ax.bar(x - w/2, faith, w, label="faithfulness (vs context)", color="#4C72B0")
ax.bar(x + w/2, corr,  w, label="correctness (vs gold)",    color="#C44E52")
for i in range(len(order)):
    ax.text(i - w/2, faith[i] + 0.02, f"{faith[i]:.2f}", ha="center", fontsize=9)
    ax.text(i + w/2, corr[i] + 0.02, f"{corr[i]:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(order); ax.set_ylim(0, 1.05)
ax.set_title("Two axes by retriever: intrinsic faithfulness vs answer correctness")
ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_faithfulness_vs_correctness.png", dpi=150); plt.show()

C:\Users\anshu\AppData\Local\Temp\ipykernel_20732\2373372883.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_faithfulness_vs_correctness.png", dpi=150); plt.show()


**Figure name: Two-axis evaluation by retriever**

Mean faithfulness (vs retrieved context) and correctness (vs gold chunk) for each retriever over the 170 answered generations. Faithfulness is near-ceiling and essentially flat across retrievers (0.97/0.95/0.96), indicating the generator rarely fabricates beyond its context. Correctness is markedly lower and separates the retrievers (0.71/0.61/0.75), with dense lowest. The gap between the blue and red bars is the central finding: grounding does not guarantee accuracy.

In [11]:
# Fig B — correctness by gold_in_context (the coupling mechanism)
fig, ax = plt.subplots(figsize=(7, 4.5))
for g_i, present in enumerate([True, False]):
    vals = [answered[(answered.retriever == r) & (answered.gold_in_context == present)]["correctness"].mean()
            for r in order]
    ax.bar(x + (g_i - 0.5) * w, vals, w, label=f"gold_in_context={present}",
           color=("#55A868" if present else "#C44E52"))
ax.set_xticks(x); ax.set_xticklabels(order); ax.set_ylim(0, 1.05)
ax.set_title("Correctness by gold_in_context (retrieval -> answer accuracy)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_correctness_by_gold_in_context.png", dpi=150); plt.show()

C:\Users\anshu\AppData\Local\Temp\ipykernel_20732\1206614210.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_correctness_by_gold_in_context.png", dpi=150); plt.show()


**Figure name: Correctness by gold-chunk presence**

Mean correctness split by whether the gold chunk was retrieved into the top-5 context. When the gold chunk is present, all three retrievers achieve ~0.80 correctness; when it is absent, correctness collapses to 0.25–0.50. This demonstrates that answer accuracy is gated by retrieval: the generator is reliable only when handed the right evidence, regardless of retriever.

In [12]:
# Fig C - KEYSTONE: query disposition recoloured by correctness.
# Each retriever's 60 queries split into 5 outcomes; the RED wedge (gold missing & incorrect)
# is the retrieval-induced error surface — expected largest for dense.
def disposition(r):
    g = scored[scored.retriever == r]
    correct = g["correctness"] >= 0.5
    pc = int(((g.gold_in_context) & (~g.abstained) & correct).sum())     # present & correct
    pi = int(((g.gold_in_context) & (~g.abstained) & (~correct)).sum())  # present & incorrect
    mc = int(((~g.gold_in_context) & (~g.abstained) & correct).sum())    # missing & correct (sibling saved)
    mi = int(((~g.gold_in_context) & (~g.abstained) & (~correct)).sum()) # missing & incorrect (retrieval error)
    ab = int(g.abstained.sum())
    return pc, pi, mc, mi, ab

segs = np.array([disposition(r) for r in order])  # rows=retriever, cols=5
labels = ["gold present & correct", "gold present & incorrect",
          "gold missing & correct (sibling)", "gold MISSING & INCORRECT (retrieval error)", "abstained"]
colors = ["#2E7D32", "#A5D6A7", "#FFB74D", "#C62828", "#B0B0B0"]
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bottom = np.zeros(len(order))
for j in range(5):
    ax.bar(order, segs[:, j], bottom=bottom, color=colors[j], label=labels[j])
    for i in range(len(order)):
        if segs[i, j] > 0:
            ax.text(i, bottom[i] + segs[i, j] / 2, str(segs[i, j]), ha="center", va="center",
                    color=("white" if j in (0, 3, 4) else "black"), fontsize=9,
                    fontweight=("bold" if j == 3 else "normal"))
    bottom += segs[:, j]
ax.set_ylabel("queries (of 60)"); ax.set_ylim(0, 64)
ax.set_title("Phase 8 - query disposition by correctness (k=5, n=60)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=1, frameon=False, fontsize=8.5)
plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_disposition_by_correctness.png", dpi=150, bbox_inches="tight"); plt.show()
print("retrieval-induced-error wedge (gold MISSING & incorrect):",
      {r: int(disposition(r)[3]) for r in order})

retrieval-induced-error wedge (gold MISSING & incorrect): {'bm25': 6, 'dense': 11, 'hybrid': 4}


C:\Users\anshu\AppData\Local\Temp\ipykernel_20732\3009432758.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "phase08_disposition_by_correctness.png", dpi=150, bbox_inches="tight"); plt.show()


**Figure name: Query disposition by correctness**

The 60 queries per retriever partitioned into five outcomes. The red "gold-missing & incorrect" wedge - the retrieval-induced error surface - is largest for dense (11 vs 6 and 4), visualising why dense yields the most faithful-but-wrong answers. The green base (gold-present & correct) is the success region; the orange band shows cases rescued by sibling chunks.

## 8. Qualitative - the faithful-but-incorrect cases (the thesis money quotes)

In [13]:
# High faithfulness but low correctness = grounded in the WRONG retrieved chunk.
fbi = answered[(answered.faithfulness >= 0.5) & (answered.correctness <= 0.0)]
print(f"{len(fbi)} answers are faithful-to-context but INCORRECT vs gold. By retriever:")
print(fbi.groupby("retriever").size().to_string()); print()
for _, r in fbi.sort_values("retriever").head(6).iterrows():
    print(f"[{r.retriever}/{r.source}] {r.query_id} | faithfulness={r.faithfulness} "
          f"correctness={r.correctness} gold_in_context={r.gold_in_context}")
    print("   correctness note:", str(r.correctness_note)[:160]); print()

25 answers are faithful-to-context but INCORRECT vs gold. By retriever:
retriever
bm25       8
dense     11
hybrid     6

[bm25/earnings] q000 | faithfulness=1.0 correctness=0.0 gold_in_context=False
   correctness note: The answer incorrectly attributes the closing remarks to CEO Rick Smith instead of Michael Petters. It also includes details not present in the reference, such 

[bm25/earnings] q013 | faithfulness=0.5 correctness=0.0 gold_in_context=False
   correctness note: The reference states a 90% free cash flow conversion target, not 66% or 60%.

[bm25/earnings] q014 | faithfulness=1.0 correctness=0.0 gold_in_context=True
   correctness note: The reference does not provide any information about whether the banker departures were consistent with management's expectations or any specific retention stat

[bm25/earnings] q025 | faithfulness=0.75 correctness=0.0 gold_in_context=False
   correctness note: The answer incorrectly states that segment margin expanded by 360 basis points t

## 9. HITL - build, hand-score THREE axes, then measure agreement

Run the next cell to write `notes/phase_08_hitl_sheet.csv`. For each row read
`gold_chunk_text` / `context_text` / `answer` and fill **`human_faithful`** (grounded in the
context? 1/0), **`human_correct`** (factually right vs the gold chunk? 1/0) and
**`human_relevant`** (addresses the question? 1/0). Save, then run the agreement cell.

In [14]:
from src.evaluation.evaluator import build_hitl_sheet
hitl = build_hitl_sheet(scored, gen_df, gold_text_by_chunk, n_per_cell=3, seed=42)
hitl.to_csv(HITL_PATH, index=False, encoding="utf-8-sig")
print(f"wrote {HITL_PATH} ({len(hitl)} items). Fill human_faithful / human_correct / human_relevant.")
hitl[["query_id","retriever","source","gold_in_context","faithfulness","correctness","relevance"]]

wrote d:\General IT\AI-ML-LJMU\final_thesis\code\notes\phase_08_hitl_sheet.csv (18 items). Fill human_faithful / human_correct / human_relevant.


,query_id,retriever,source,gold_in_context,faithfulness,correctness,relevance
0,q008,bm25,earnings,True,1.000000,1.0,1.0
1,q009,bm25,earnings,True,1.000000,1.0,1.0
2,q027,bm25,earnings,False,1.000000,0.0,1.0
3,q039,bm25,edgar,True,1.000000,0.0,1.0
4,q040,bm25,edgar,True,0.750000,0.5,1.0
5,q057,bm25,edgar,False,1.000000,0.5,1.0
6,q009,dense,earnings,True,1.000000,1.0,1.0
7,q010,dense,earnings,True,1.000000,1.0,1.0
8,q014,dense,earnings,True,1.000000,0.0,1.0
9,q045,dense,edgar,False,1.000000,1.0,1.0


In [15]:
# RUN ONLY AFTER filling the human_* columns.
from src.evaluation.evaluator import human_judge_agreement
filled = pd.read_csv(HITL_PATH, encoding="utf-8-sig")
agree = human_judge_agreement(filled)
print(json.dumps(agree, indent=2))
summ = json.loads(SUMMARY_PATH.read_text())
summ["hitl"] = {"n_items": int(len(filled)), "agreement": agree}
SUMMARY_PATH.write_text(json.dumps(summ, indent=2)); print("updated", SUMMARY_PATH)

{
  "faithfulness": {
    "n": 18,
    "agreement": 0.8333,
    "cohen_kappa": 0.0
  },
  "correctness": {
    "n": 18,
    "agreement": 0.7222,
    "cohen_kappa": 0.1176
  },
  "relevance": {
    "n": 18,
    "agreement": 1.0,
    "cohen_kappa": 1.0
  }
}
updated d:\General IT\AI-ML-LJMU\final_thesis\code\notes\phase_08_summary.json


What HITL adds vs the automated metrics. Your automated judge (gpt-4o) scored all 170 answers. But an examiner can fairly ask: "why should I trust an AI grading another AI?" HITL — Human In The Loop — is your answer. You hand-scored 18 of those same answers yourself, blind to what the judge said, on the same yes/no questions. Then you compare: when you and the judge looked at the same answer, did you reach the same verdict? If yes most of the time, the judge is trustworthy and the other 152 scores you didn't check by hand are credible too. So HITL doesn't add new measurements — it validates the automated ones. It turns "an AI said so" into "an AI whose judgments match a human's said so." That's the whole purpose.

<hr>

What Cohen's κ (kappa) is. Imagine you and the judge agree on 13 of 18 correctness labels — that's 72% raw agreement. Sounds decent. But here's the catch: if most answers are "correct," then two people who both just guessed "correct" every time would also agree most of the time, purely by luck. Cohen's κ corrects for that luck. It asks: "how much better is the real agreement than the agreement you'd expect from chance alone?"

<hr>

The formula in words: κ = (actual agreement − chance agreement) / (perfect agreement − chance agreement). κ = 1 means perfect agreement; κ = 0 means "no better than two people guessing independently"; below 0 means worse than chance. Landis & Koch's rough labels: >0.8 excellent, 0.6–0.8 substantial, 0.4–0.6 moderate, 0.2–0.4 fair, <0.2 slight. So your correctness κ of 0.12 reads as "slight," and that sounds bad.

<hr>

What the kappa paradox is (and why it's not actually bad here). κ has a known failure mode: when nearly everything falls into one category, the "chance agreement" term becomes huge, and κ gets crushed toward 0 even when raw agreement is high.

<hr>


Concrete example with your faithfulness axis: the judge marked all 18 answers "faithful." You marked 15 of 18 "faithful." You agreed on 15 — that's 83%. But because the judge never said "unfaithful" even once, the math says "of course you mostly agree — one of you always says the same thing, so agreeing is trivially easy." So κ computes to 0.0, not because the judge is random, but because there was almost no disagreement possible to measure. That's the paradox: high agreement, near-zero κ, purely because the answers are lopsided (almost everything is faithful — which is itself your finding!). The same thing, milder, drags down your correctness κ (you both marked ~14–15 of 18 "correct," so the data is lopsided there too).

--------------
So the honest reading of your HITL: report the raw agreement (83% / 72% / 100%) as the real signal, and explain that κ is artificially low because the answers are lopsided. This is a well-documented statistical effect with citations you can point to — it's not hand-waving.